# 07a. Logistic Regression Hyperparameter Tuning

This notebook performs a lightweight, reproducible `RandomizedSearchCV` for the Logistic Regression baseline. It uses only the training split for selection, optimizes PR-AUC for the imbalanced default class, and evaluates the selected pipeline once on the held-out test set.

## 1. Setup and load the unchanged split

In [ ]:
from pathlib import Path
import json
import sys

current = Path.cwd()
while current != current.parent:
    if (current / 'src').is_dir():
        PROJECT_ROOT = current
        break
    current = current.parent
else:
    raise RuntimeError("Could not find project root containing 'src'.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import pandas as pd
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from src.config import RANDOM_STATE, TARGET
from src.data.data_cleaning import clean_and_split
from src.models.logistic_regression import build_logistic_regression_pipeline

train_df, test_df = clean_and_split()
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

RESULTS_DIR = PROJECT_ROOT / 'experiments' / 'results'
MODELS_DIR = PROJECT_ROOT / 'models' / 'tuned'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Train:', X_train.shape, 'default rate:', round(y_train.mean(), 4))
print('Test: ', X_test.shape, 'default rate:', round(y_test.mean(), 4))

## 2. Configure a small randomized search

The search tests four randomly selected configurations with stratified 3-fold CV (12 fits total). `C` controls regularization, `class_weight` tests imbalance handling, and `solver` compares two L2-compatible optimizers. PR-AUC is the refit metric; ROC-AUC and F1 are retained for interpretation.

In [ ]:
pipeline = build_logistic_regression_pipeline()
parameter_distributions = {
    'model__C': [0.01, 0.1, 1.0, 10.0],
    'model__class_weight': [None, 'balanced'],
    'model__solver': ['lbfgs', 'liblinear'],
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=parameter_distributions,
    n_iter=4,
    scoring={'pr_auc': 'average_precision', 'roc_auc': 'roc_auc', 'f1': 'f1'},
    refit='pr_auc',
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    return_train_score=True,
    verbose=2,
)
print('Search workload: 4 configurations x 3 folds = 12 fits')

## 3. Run the search

Only the training split is supplied here. The held-out test set remains untouched until the best hyperparameters have been fixed.

In [ ]:
random_search.fit(X_train, y_train)
print('Best mean CV PR-AUC:', round(random_search.best_score_, 6))
print('Best parameters:', random_search.best_params_)

## 4. Inspect and save the search evidence

In [ ]:
search_results = pd.DataFrame(random_search.cv_results_)
search_summary = search_results[[
    'rank_test_pr_auc', 'param_model__C', 'param_model__class_weight',
    'param_model__solver', 'mean_test_pr_auc', 'std_test_pr_auc',
    'mean_test_roc_auc', 'std_test_roc_auc', 'mean_test_f1',
    'mean_train_pr_auc', 'mean_fit_time',
]].sort_values('rank_test_pr_auc')
search_summary.to_csv(
    RESULTS_DIR / 'logistic_regression_random_search.csv', index=False
)

best_parameters = {
    key.replace('model__', ''): value
    for key, value in random_search.best_params_.items()
}
with open(RESULTS_DIR / 'logistic_regression_best_parameters.json', 'w', encoding='utf-8') as file:
    json.dump(best_parameters, file, indent=2)
search_summary

## 5. Final held-out test evaluation

`best_estimator_` is already refitted on the complete training split. The test set is now used once with the standard 0.5 threshold.

In [ ]:
best_pipeline = random_search.best_estimator_
test_probability = best_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, test_prediction, labels=[0, 1]).ravel()

test_results = pd.DataFrame([{
    'model_name': 'Logistic Regression Tuned',
    'threshold': 0.5,
    'accuracy': accuracy_score(y_test, test_prediction),
    'roc_auc': roc_auc_score(y_test, test_probability),
    'pr_auc': average_precision_score(y_test, test_probability),
    'recall': recall_score(y_test, test_prediction),
    'precision': precision_score(y_test, test_prediction, zero_division=0),
    'f1': f1_score(y_test, test_prediction, zero_division=0),
    'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
}])
test_results.to_csv(RESULTS_DIR / 'logistic_regression_tuned_test.csv', index=False)
test_results

## 6. Save the selected pipeline

In [ ]:
model_path = MODELS_DIR / 'logistic_regression_tuned.pkl'
joblib.dump(best_pipeline, model_path)
print('Saved tuned pipeline:', model_path)

## 7. Interpretation

After execution, compare the selected model's CV PR-AUC and ROC-AUC with the baseline results. Report an improvement only if the executed evidence supports it. Threshold optimization is intentionally outside this notebook's scope.